# Task 2.4 — Aggregate IoT Sensor Data — 15-Minute Zone Occupancy

15-minute tumbling window aggregations per `zone_id`:
- `avg_queue_length`, `max_queue_length`, `avg_wait_time_mins`, `cumulative_pax_through`
- `zone_utilization_pct`: avg_queue / zone_rated_capacity (hardcoded capacity lookup)
- Flag **CONGESTED** if `avg_wait_time_mins > 20` in any 15-minute bucket
- Write to `silver.zone_occupancy_15min` — base data for security prediction model


In [ ]:
# ── Imports & Configuration ────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

DATA_DIR = "/FileStore/airport_analytics_data"
BRONZE_DIR = "/FileStore/delta_lake/bronze"
SILVER_DIR = "/FileStore/delta_lake/silver"

try:
    spark
except NameError:
    spark = (SparkSession.builder
        .appName("Task_2_4_Zone_Occupancy")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate())


## Step 1 — Read Bronze IoT Counters

In [ ]:
# ── Read Bronze IoT counter data ──────────────────────────────────
iot_df = spark.read.format("delta").load(f"{BRONZE_DIR}/iot_counters")

print(f"IoT records: {iot_df.count()}")
print(f"Unique zones: {iot_df.select('zone_id').distinct().count()}")
iot_df.show(5, truncate=False)


## Step 2 — 15-Minute Window Aggregation

In [ ]:
# ── 15-minute tumbling window aggregation per zone_id ─────────────
# Create 15-min time bucket
zone_agg_df = (iot_df
    .withColumn("timestamp", F.to_timestamp("timestamp"))
    .withColumn("time_bucket",
        F.window("timestamp", "15 minutes").start
    )
)

zone_15min_df = (zone_agg_df
    .groupBy("zone_id", "terminal", "location_type", "time_bucket")
    .agg(
        F.round(F.avg("queue_length_estimate"), 1).alias("avg_queue_length"),
        F.max("queue_length_estimate").alias("max_queue_length"),
        F.round(F.avg("avg_wait_time_minutes"), 1).alias("avg_wait_time_mins"),
        F.sum("pax_count_in").alias("cumulative_pax_in"),
        F.sum("pax_count_out").alias("cumulative_pax_out"),
        F.count("*").alias("sensor_readings")
    )
)

# Cumulative pax through
zone_15min_df = zone_15min_df.withColumn(
    "cumulative_pax_through",
    F.col("cumulative_pax_in") + F.col("cumulative_pax_out")
)

print(f"15-min zone buckets: {zone_15min_df.count()}")
zone_15min_df.show(5, truncate=False)


## Step 3 — Zone Capacity Lookup & Congestion Flag

In [ ]:
# ── Zone rated capacity lookup (hardcoded) ────────────────────────
# Capacity by location type (passengers per zone)
ZONE_CAPACITY = {
    "SECURITY_LANE": 80,
    "IMMIGRATION": 100,
    "GATE_AREA": 150,
    "CHECK_IN_ZONE": 120,
    "BAGGAGE_CLAIM": 100
}

zone_cap_df = spark.createDataFrame(
    [(k, v) for k, v in ZONE_CAPACITY.items()],
    ["location_type", "zone_rated_capacity"]
)

zone_15min_df = zone_15min_df.join(zone_cap_df, on="location_type", how="left")

# Compute utilization percentage
zone_15min_df = zone_15min_df.withColumn(
    "zone_utilization_pct",
    F.round(
        (F.col("avg_queue_length") / F.col("zone_rated_capacity")) * 100, 1
    )
)

# Flag CONGESTED if avg_wait_time_mins > 20
zone_15min_df = zone_15min_df.withColumn(
    "congestion_flag",
    F.when(F.col("avg_wait_time_mins") > 20, "CONGESTED").otherwise("NORMAL")
)

# Add processing metadata
zone_15min_df = zone_15min_df.withColumn("processing_ts", F.current_timestamp())

print("\nCongestion Distribution:")
zone_15min_df.groupBy("congestion_flag").count().show()
print("Utilization by Location Type:")
zone_15min_df.groupBy("location_type").agg(
    F.round(F.avg("zone_utilization_pct"), 1).alias("avg_utilization_pct")
).show()


## Step 4 — Write to Silver Delta

In [ ]:
# ── Write silver.zone_occupancy_15min ─────────────────────────────
silver_zone_path = f"{SILVER_DIR}/zone_occupancy_15min"

(zone_15min_df.write
    .format("delta")
    .mode("overwrite")
    .save(silver_zone_path))

print(f"Written to {silver_zone_path}")
result_df = spark.read.format("delta").load(silver_zone_path)
print(f"Silver zone_occupancy_15min total: {result_df.count()}")
result_df.orderBy("time_bucket").show(10, truncate=False)
